# DSPIN models — program-level fit per `(tissue, supertype)`

Fits **program-level** DSPIN on interactive pilot slices from `pilot_manifest.csv`.

- `SMOKE_TEST=True`: one slice, `num_spin=15`, `num_repeat=3`
- Full interactive: `num_spin=20`, `num_repeat=10`
- Method: `mcmc_maximum_likelihood`, fallback `pseudo_likelihood`
- Writes CSV/PNG artifact contract for `0X_DSPIN_Analysis.qmd`


In [ ]:
from pathlib import Path
import json
import time
import traceback
from datetime import datetime, timezone

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

WKDIR = Path("/resnick/groups/mthomson/jboktor/WILDRxSPF_brains")
DSPIN_DIR = WKDIR / "data/interim/dspin"
FIG_DIR = WKDIR / "figures/DSPIN/snRNASeq"
FIG_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST = DSPIN_DIR / "pilot_manifest.csv"

# Flags
SMOKE_TEST = True          # set False for full interactive freeze (≤5)
RUN_OPTIONAL_SBATCH = False  # ranks 6–15 only if you exported those h5ads

NUM_SPIN_SMOKE = 15
NUM_REPEAT_SMOKE = 3
NUM_SPIN_FULL = 20
NUM_REPEAT_FULL = 10

print("SMOKE_TEST=", SMOKE_TEST)


In [ ]:
try:
    from dspin.dspin import DSPIN
    import dspin.plot as dp
    print("dspin import OK")
except Exception as e:
    raise SystemExit(
        "Install dspin in this kernel env: pip install dspin\n" + repr(e)
    )


In [ ]:
manifest = pd.read_csv(MANIFEST)
if SMOKE_TEST:
    run_df = manifest.iloc[[0]].copy()
elif RUN_OPTIONAL_SBATCH:
    run_df = manifest.copy()
else:
    run_df = manifest.query("run_tier == 'interactive'").copy() if "run_tier" in manifest else manifest.head(5)
print(run_df[["tissue", "supertype_name"]].to_string(index=False) if "tissue" in run_df else run_df.head())


## Fit helper — save R-readable artifacts


In [ ]:
def slice_dir(row) -> Path:
    if "slice_path" in row and pd.notna(row["slice_path"]):
        return Path(row["slice_path"])
    return DSPIN_DIR / f"{row['tissue']}__{row['supertype_slug']}"


def matrix_to_tidy(mat, row_names, col_names, value_name):
    arr = np.asarray(mat)
    df = pd.DataFrame(arr, index=list(row_names)[: arr.shape[0]], columns=list(col_names)[: arr.shape[1]])
    return (
        df.rename_axis("program_id")
        .reset_index()
        .melt(id_vars="program_id", var_name="sample_id", value_name=value_name)
    )


def save_network_preview(network, program_names, out_png: Path, title: str):
    fig, ax = plt.subplots(figsize=(8, 7))
    M = np.asarray(network, dtype=float)
    vmax = np.nanmax(np.abs(M)) if M.size else 1.0
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0
    im = ax.imshow(M, cmap="coolwarm", vmin=-vmax, vmax=vmax)
    ax.set_title(title)
    labels = [str(p)[:18] for p in program_names]
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.set_yticklabels(labels, fontsize=7)
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=150)
    plt.close(fig)


def ensure_leiden(adata: ad.AnnData) -> ad.AnnData:
    """ProgramDSPIN.gene_program_discovery defaults to cluster_key='leiden'."""
    if "leiden" in adata.obs.columns:
        return adata
    n_comps = min(30, max(2, adata.n_obs - 1), max(2, adata.n_vars - 1))
    sc.pp.pca(adata, n_comps=n_comps)
    sc.pp.neighbors(adata, n_neighbors=min(15, max(2, adata.n_obs - 1)))
    try:
        sc.tl.leiden(adata, key_added="leiden", flavor="igraph", directed=False, n_iterations=2)
    except TypeError:
        sc.tl.leiden(adata, key_added="leiden")
    return adata


def fit_one(row, num_spin: int, num_repeat: int) -> dict:
    sdir = slice_dir(row)
    h5ad = sdir / "filtered.h5ad"
    if not h5ad.exists():
        h5ad = sdir / "raw_counts.h5ad"
    if not h5ad.exists():
        return {"status": "error", "error": f"missing h5ad under {sdir}"}

    t0 = time.time()
    adata = ad.read_h5ad(h5ad)
    if np.asarray(adata.X.max() if not hasattr(adata.X, 'max') else adata.X.max()) > 50:
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
    adata = ensure_leiden(adata)

    save_path = str(sdir / "dspin_run")
    Path(save_path).mkdir(parents=True, exist_ok=True)

    status = {
        "tissue": row["tissue"],
        "supertype_name": row["supertype_name"],
        "num_spin": num_spin,
        "num_repeat": num_repeat,
        "n_cells": int(adata.n_obs),
        "n_genes": int(adata.n_vars),
        "n_SPF": int((adata.obs["microbiome"].astype(str) == "SPF").sum()),
        "n_WildR": int((adata.obs["microbiome"].astype(str) == "WildR").sum()),
        "started": datetime.now(timezone.utc).isoformat(),
        "status": "running",
        "method": None,
        "error": None,
    }

    model = DSPIN(adata, save_path, num_spin=num_spin)
    model.gene_program_discovery(num_repeat=num_repeat, seed=0, cluster_key="leiden")

    try:
        model.network_inference(sample_id_key="sample_id", method="auto")
        status["method"] = "auto"
    except Exception as e:
        print("auto inference failed, falling back to pseudo_likelihood:", e)
        model.network_inference(sample_id_key="sample_id", method="pseudo_likelihood")
        status["method"] = "pseudo_likelihood"

    model.response_relative_to_control(
        sample_id_key="sample_id",
        if_control_key="if_control",
        batch_key="batch",
    )

    program_names = list(getattr(model, "name_list_short", None) or [f"P{i}" for i in range(num_spin)])
    sample_list = list(model.sample_list)

    rel = np.asarray(model.relative_responses)
    if rel.ndim == 2 and rel.shape[0] == len(sample_list):
        tidy_rel = matrix_to_tidy(rel.T, program_names, sample_list, "relative_h")
    elif rel.ndim == 2 and rel.shape[1] == len(sample_list):
        tidy_rel = matrix_to_tidy(rel, program_names, sample_list, "relative_h")
    else:
        tidy_rel = matrix_to_tidy(
            np.atleast_2d(rel),
            program_names[: np.atleast_2d(rel).shape[0]],
            [f"c{j}" for j in range(np.atleast_2d(rel).shape[1])],
            "relative_h",
        )
    tidy_rel.to_csv(sdir / "relative_responses.csv", index=False)

    resp = np.asarray(model.responses)
    if resp.ndim == 2 and resp.shape[0] == len(sample_list):
        matrix_to_tidy(resp.T, program_names, sample_list, "h").to_csv(sdir / "responses.csv", index=False)
    elif resp.ndim == 2:
        matrix_to_tidy(resp, program_names[: resp.shape[0]], sample_list[: resp.shape[1]], "h").to_csv(
            sdir / "responses.csv", index=False
        )

    net = np.asarray(model.network)
    net_df = pd.DataFrame(net, index=program_names[: net.shape[0]], columns=program_names[: net.shape[1]])
    net_df.to_csv(sdir / "network.csv")

    prog_gene_path = sdir / "programs_genes.csv"
    rows = []
    full_names = list(getattr(model, "name_list", program_names))
    for pname, full in zip(program_names, full_names):
        if "-" in str(full):
            genes = str(full).split("-", 1)[1].split(",")
            for g in genes:
                if g and g != "nan":
                    rows.append({"program_id": pname, "gene": g, "weight": np.nan})
    pd.DataFrame(rows).to_csv(prog_gene_path, index=False)

    sample_meta = (
        adata.obs.groupby(["sample_id", "microbiome", "batch", "if_control"], observed=True)
        .size()
        .reset_index(name="n_cells")
    )
    sample_meta.to_csv(sdir / "sample_meta.csv", index=False)

    png = sdir / "network_preview.png"
    save_network_preview(net, list(net_df.index), png, title=f"{row['tissue']} | {row['supertype_name']}")
    fig_copy = FIG_DIR / f"{row['tissue']}__{row['supertype_slug']}_network_preview.png"
    fig_copy.write_bytes(png.read_bytes())

    status["status"] = "ok"
    status["elapsed_s"] = round(time.time() - t0, 1)
    status["finished"] = datetime.now(timezone.utc).isoformat()
    (sdir / "fit_status.json").write_text(json.dumps(status, indent=2))
    return status


def run_row(row):
    num_spin = NUM_SPIN_SMOKE if SMOKE_TEST else NUM_SPIN_FULL
    num_repeat = NUM_REPEAT_SMOKE if SMOKE_TEST else NUM_REPEAT_FULL
    try:
        return fit_one(row, num_spin=num_spin, num_repeat=num_repeat)
    except Exception as e:
        sdir = slice_dir(row)
        sdir.mkdir(parents=True, exist_ok=True)
        err = {
            "status": "error",
            "tissue": row.get("tissue"),
            "supertype_name": row.get("supertype_name"),
            "error": f"{type(e).__name__}: {e}",
            "traceback": traceback.format_exc(),
        }
        (sdir / "fit_status.json").write_text(json.dumps(err, indent=2))
        print("ERROR", row.get("supertype_name"), e)
        return err


In [ ]:
results = []
for _, row in run_df.iterrows():
    print("=" * 60)
    print("Fitting", row["tissue"], row["supertype_name"])
    results.append(run_row(row))

res_df = pd.DataFrame(results)
res_df.to_csv(DSPIN_DIR / "fit_summary.csv", index=False)
display(res_df)
